# 06. Custom exceptions

**Tier:** Intermediate. **Why you'd use this:** A negative age is not a generic `ValueError` you want mixed with every other bad value. A small exception class names the failure so callers can catch it precisely.

**After this lesson you can:** subclass `Exception`; raise and catch your type; choose a narrow `except`.
About 25 minutes. **Uses:** [Inheritance](03-inheritance-lesson.ipynb), [Raise](../06-errors/02-raise-lesson.ipynb)


## Quick review

1. Why are enum members safe to compare with `is`?
2. What does `Color("missing")` do?

<details><summary>Answers</summary>

They are singletons. It raises `ValueError`.
</details>


## The idea

Define a class that inherits from `Exception`. Raise it with `raise YourError("message")`. Catch it with `except YourError`. Callers can handle your failure without catching every `Exception`.

<details><summary>Explain more</summary>

Inheritance decides what an `except` matches. A subclass is caught by an `except` for its base.
</details>


## See it work

Reject a negative age with a named error.


In [1]:
class NegativeAgeError(Exception):
    pass


def require_age(age):
    # 1. Raise a specific type callers can catch.
    if age < 0:
        raise NegativeAgeError("negative")
    return age


print(require_age(3))
try:
    require_age(-1)
except NegativeAgeError as error:
    print(type(error).__name__, str(error))


3
NegativeAgeError negative


## Check yourself

Answer these before you run the cell.

1. Does `except Exception` catch `NegativeAgeError`?
2. Does `except ValueError` catch it if your class subclasses `Exception` only?


In [2]:
class NegativeAgeError(Exception):
    pass


try:
    raise NegativeAgeError("negative")
except Exception as error:
    print(type(error).__name__)


NegativeAgeError


<details><summary>Answers</summary>

Yes. No. `ValueError` is a sibling branch, not a base, unless you inherit from it.
</details>


## Why it works this way

Custom exceptions are ordinary subclasses of `Exception`. `except NegativeAgeError` catches that class and its subclasses through the same MRO rules you use for inheritance. Raising a specific type lets callers recover narrowly without swallowing unrelated failures. Deep dive: [Attribute lookup](../../internals/08-attributes/01-lookup-lesson.ipynb). Related: [Raise](../06-errors/02-raise-lesson.ipynb).

A bare `except Exception` still catches your custom type because it sits under `Exception`. That is useful for a top-level boundary and harmful inside helpers that should only handle one failure mode. Name the exception after the problem the caller can act on. Keep the hierarchy shallow until you have a real taxonomy.

Cost is one class object and the usual traceback allocation when raised. The design reason is structured failure signals: types are part of the API. String messages are for humans; types are for code.

Evidence: `except NegativeAgeError` handles `raise NegativeAgeError(...)`, and a parent `except Exception` would also match, while `except ValueError` would not unless you inherit from it on purpose.


Empty exception subclasses are normal at first. Add attributes later when callers need structured data beyond the message. Keep messages short and stable enough to test. Prefer raising from the layer that detected the problem, and catch at the layer that can decide a recovery. That separation is the same EAFP idea from the errors unit, now with a type you own. If two modules share a failure mode, put the exception class in a small shared module instead of duplicating it.


In [3]:
class NegativeAgeError(Exception):
    pass


try:
    raise NegativeAgeError("negative")
except NegativeAgeError as error:
    print(type(error).__name__, str(error))

print(issubclass(NegativeAgeError, Exception))
print(issubclass(NegativeAgeError, ValueError))


NegativeAgeError negative
True
False


## When to use it

Use a custom exception when callers need to distinguish this failure from others. Inherit from `Exception` or from a more specific standard type when that match is intentional. Don't raise bare strings. Don't catch `Exception` in a small helper that should only handle your type. Document which exceptions are part of the public contract so callers know what to catch.


## Common mistake

Catching `Exception` around a narrow check and hiding bugs that are not age errors.


In [4]:
class NegativeAgeError(Exception):
    pass


def require_age(age):
    try:
        if age < 0:
            raise NegativeAgeError("negative")
        return age
    except Exception:
        return 0


print(require_age(-1))
print(require_age("x"))


0
0


Catch only `NegativeAgeError`, or let unexpected errors propagate.


In [5]:
class NegativeAgeError(Exception):
    pass


def require_age(age):
    if age < 0:
        raise NegativeAgeError("negative")
    return age


try:
    print(require_age(-1))
except NegativeAgeError:
    print("rejected")


rejected


## Try it

Create `AgeError(Exception)` and `NegativeAgeError(AgeError)`. Which `except` clauses catch a negative age, and why might a library expose the base type?


## Recap

- Custom exceptions are subclasses of `Exception`.
- `except` matches by inheritance.
- Narrow catches keep real bugs visible.

**Next:** [practice](06-custom-exceptions-practice.ipynb)
